In [ ]:
"""Visualization and Scene tutorial."""

# Visualization and Scene.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeepMIMO/DeepMIMO/blob/main/docs/tutorials/2_visualization.py)
&nbsp;
[![GitHub](https://img.shields.io/badge/Open_on-GitHub-181717?logo=github&style=for-the-badge)](https://github.com/DeepMIMO/DeepMIMO/blob/main/docs/tutorials/2_visualization.py)

---

**Tutorial Overview:**
- Coverage Maps - Visualizing signal coverage
- Rays - Ray propagation visualization
- Path Plots - Visualization of different path components
- Scene & Materials - 3D scene and material visualization
- Plot Overlays - Combining different visualizations

**Related Video:** [Visualization Video](https://youtu.be/MO7h2shBhsc)

---

In [ ]:
# Import libraries
import matplotlib.pyplot as plt
import numpy as np

import deepmimo as dm

In [ ]:
# Load dataset
scen_name = "asu_campus_3p5"
dm.download(scen_name)
dataset = dm.load(scen_name)

## Coverage Maps

Visualize signal coverage across the scenario.

In [ ]:
# Plot power coverage
dataset.power.plot()
plt.title("Power Coverage Map")
plt.show()

In [ ]:
# Plot pathloss coverage
dataset.pathloss.plot()
plt.title("Pathloss Coverage Map")
plt.show()

## Rays

Visualize ray propagation paths.

In [ ]:
# Plot rays for a user with line-of-sight
u_idx = np.where(dataset.los == 1)[0][100]
dataset.plot_rays(u_idx, proj_3D=False, dpi=100)
plt.title("Ray Propagation Paths")
plt.show()

## Path Plots

Visualize different path components and characteristics.

### Percentage of Power

In [ ]:
# Plot power percentage per path
if hasattr(dataset, "power"):
    # Calculate power percentage
    linear_power = 10 ** (dataset.power / 10)
    total_power = np.sum(linear_power, axis=1, keepdims=True)
    power_pct = (linear_power / (total_power + 1e-20)) * 100

    plt.figure(figsize=(10, 6))
    plt.hist(power_pct[power_pct > 0].flatten(), bins=50)
    plt.xlabel("Power Percentage (%)")
    plt.ylabel("Count")
    plt.title("Distribution of Power Percentage per Path")
    plt.show()

### Number of Interactions

In [ ]:
# Visualize number of interactions per path
if hasattr(dataset, "interactions"):
    interactions = dataset.interactions
    # Count digits in interaction codes
    num_interactions = np.array(
        [[len(str(int(x))) if x > 0 else 0 for x in row] for row in interactions]
    )

    plt.figure(figsize=(10, 6))
    plt.hist(num_interactions.flatten(), bins=range(10))
    plt.xlabel("Number of Interactions")
    plt.ylabel("Count")
    plt.title("Distribution of Interaction Counts")
    plt.show()

## Scene & Materials

Explore the 3D scene and materials.

### Scene Visualization

In [ ]:
# Plot the 3D scene
if hasattr(dataset, "scene"):
    dataset.scene.plot()
    plt.title("3D Scene")
    plt.show()
else:
    print("Scene data not available for this scenario")

### Materials

In [ ]:
# Display material properties
if hasattr(dataset, "materials"):
    print("Available Materials:")
    print(dataset.materials)
else:
    print("Material data not available for this scenario")

## Plot Overlays

Combine different visualizations for comprehensive analysis.

### 2D Scene, Coverage & Rays Overlay

In [ ]:
# Overlay coverage map with rays
fig, ax = plt.subplots(figsize=(12, 8))

# Plot coverage as background
dataset.power.plot(ax=ax)

# Overlay rays for a selected user
los_users = np.where(dataset.los == 1)[0]
if len(los_users) > 0:
    dataset.plot_rays(los_users[50], ax=ax, proj_3D=False)

plt.title("Coverage Map with Ray Overlay")
plt.show()

### 3D Scene & Rays Overlay

In [ ]:
# 3D visualization with scene and rays
if hasattr(dataset, "scene"):
    # Plot rays first, then overlay the scene
    ax = dataset.plot_rays(los_users[50])
    dataset.scene.plot(ax=ax)
    ax.legend().set_visible(False)

    plt.title("3D Scene with Rays")
    plt.show()
else:
    print("3D scene not available for this scenario")

## Multipath Lifetime Map (MPLM)

Visualize spatial regions with similar multipath characteristics.
The MPLM colors each receiver location based on its unique multipath signature.
Receivers with the same combination of propagation paths will have the same color.

### Basic MPLM Visualization

In [ ]:
# Plot the Multipath Lifetime Map
dataset.plot_mplm(dpi=150)
plt.show()

The MPLM computation involves three lazy-evaluated properties:
- `dataset.inter_vec`: Vectorized interaction codes
- `dataset.path_ids`: Unique IDs for each path signature
- `dataset.path_hash`: Hash for each user's multipath mix

In [ ]:
# Access the underlying data (computed lazily on first access)
path_hash = dataset.path_hash
path_ids = dataset.path_ids

# Statistics
print(f"Total users: {dataset.n_ue}")
print(f"Active users (with paths): {(path_hash != -1).sum()}")
print(f"Unique multipath signatures: {len(np.unique(path_hash[path_hash != -1]))}")

### Analyzing Multipath Zones

In [ ]:
# Find the most common multipath signatures
active_hashes = path_hash[path_hash != -1]
unique_hashes, counts = np.unique(active_hashes, return_counts=True)
top_5_idx = np.argsort(counts)[-5:]

print("Top 5 most common multipath signatures:")
for idx in reversed(top_5_idx):
    hash_val = unique_hashes[idx]
    count = counts[idx]
    print(f"  Hash {hash_val}: {count} users ({count / len(active_hashes) * 100:.1f}%)")

---

## Next Steps

Continue with:
- **Tutorial 3: Detailed Channel Generation** - Deep dive into channel generation parameters
- **Tutorial 4: User Selection and Dataset Manipulation** - Learn how to filter and sample users